In [ ]:
class EconomyBMW:

    def __init__(self, K_1=0.0, L_1=0.0, Mh_1=0.0, Y_1=0.0, r_1=0.04):
        self.period = [0]

        #Exogenous inputs, one value stored per period.
        self.alpha0 = [0.0]
        self.alpha1 = [0.0]
        self.alpha2 = [0.0]
        self.delta = [0.0]
        self.gamma = [0.0]
        self.kappa = [0.0]
        self.pr = [1.0]
        self.r_bar = [r_1]

        #Computed flows.
        self.AF = [0.0]
        self.KT = [0.0]
        self.I = [0.0]
        self.WB = [0.0]
        self.YD = [0.0]
        self.C = [0.0]
        self.Y = [Y_1]
        self.N = [0.0]
        self.W = [0.0]
        self.dL = [0.0]
        self.dMh = [0.0]
        self.dK = [0.0]

        #Stocks (levels carried across periods).
        self.K = [K_1]
        self.L = [L_1]
        self.Mh = [Mh_1]
        self.r = [r_1]

    def runPeriod(self, alpha0, alpha1, alpha2, delta, gamma, kappa, pr, r_bar,
                  tol=1e-8, max_iter=1000):
        K_1 = self.K[-1]
        L_1 = self.L[-1]
        Mh_1 = self.Mh[-1]
        Y_1 = self.Y[-1]
        r_1 = self.r[-1]

        #Firms: amortization funds / depreciation allowance
        AF = delta * K_1  #eq. 7.7 / eq. 7.18 (AF = DA)

        #Firms: investment (depends only on the previous period)
        KT = kappa * Y_1  # eq. 7.19
        I = gamma * (KT - K_1) + AF  #eq. 7.20, with eq. 7.2 (I_s = I_d)

        #Simultaneous block WB<->YD<->C<->Y
        Y = Y_1
        WB = YD = C = 0.0
        for _ in range(max_iter):
            WB = Y - r_1 * L_1 - AF                   #eq. 7.6
            YD = WB + r_1 * Mh_1                       #eq. 7.9
            C = alpha0 + alpha1 * YD + alpha2 * Mh_1    #eq. 7.16
            Y_new = C + I                               #eq. 7.5
            if abs(Y_new - Y) < tol:
                Y = Y_new
                break
            Y = Y_new
        else:
            raise RuntimeError(
                f"Period {self.period[-1] + 1}: income did not converge "
                f"after {max_iter} Gauss-Seidel iterations.")

        #Employment and wage rate (actual current Y) 
        N = Y / pr                     #eq. 7.14
        W = WB / N if N != 0 else 0.0  #eq. 7.15

        #Flows of funds
        dL = I - AF   #eq. 7.8 (with eq. 7.4, 7.11)
        dMh = YD - C  #eq. 7.10
        dK = I - AF   #eq. 7.17 (DA = AF)

        #Stock updates
        K = K_1 + dK
        L = L_1 + dL
        Mh = Mh_1 + dMh
        r = r_bar  #eq. 7.12 and eq. 7.21 (r_m = r_l = r_bar)

        self.period.append(self.period[-1] + 1)
        self.alpha0.append(alpha0)
        self.alpha1.append(alpha1)
        self.alpha2.append(alpha2)
        self.delta.append(delta)
        self.gamma.append(gamma)
        self.kappa.append(kappa)
        self.pr.append(pr)
        self.r_bar.append(r_bar)

        self.AF.append(AF)
        self.KT.append(KT)
        self.I.append(I)
        self.WB.append(WB)
        self.YD.append(YD)
        self.C.append(C)
        self.Y.append(Y)
        self.N.append(N)
        self.W.append(W)
        self.dL.append(dL)
        self.dMh.append(dMh)
        self.dK.append(dK)
        self.K.append(K)
        self.L.append(L)
        self.Mh.append(Mh)
        self.r.append(r)

    def returnCurrentState(self):
        return (self.period[-1], self.alpha0[-1], self.alpha1[-1], self.alpha2[-1],
                self.delta[-1], self.gamma[-1], self.kappa[-1], self.pr[-1],
                self.r_bar[-1], self.AF[-1], self.KT[-1], self.I[-1], self.WB[-1],
                self.YD[-1], self.C[-1], self.Y[-1], self.N[-1], self.W[-1],
                self.dL[-1], self.dMh[-1], self.dK[-1], self.K[-1], self.L[-1],
                self.Mh[-1], self.r[-1])

    def _print_table(self, indices, title, max_width=76):
        #Same plain style as Chapter 3; splits into blocks if periods overflow max_width.
        variables = [
            ("alpha0", self.alpha0), ("alpha1", self.alpha1),
            ("alpha2", self.alpha2), ("delta", self.delta),
            ("gamma", self.gamma), ("kappa", self.kappa),
            ("pr", self.pr), ("r_bar", self.r_bar),
            ("AF/DA", self.AF), ("K^T", self.KT), ("I", self.I),
            ("WB", self.WB), ("YD", self.YD), ("C", self.C), ("Y", self.Y),
            ("N", self.N), ("W", self.W), ("dL", self.dL),
            ("dMh", self.dMh), ("dK", self.dK),
            ("K", self.K), ("L", self.L), ("Mh", self.Mh), ("r", self.r),
        ]

        label_width = 8
        col_width = 10

        indices = list(indices)
        cols_per_block = max(1, (max_width - label_width) // col_width)
        blocks = [indices[i:i + cols_per_block]
                  for i in range(0, len(indices), cols_per_block)]

        for block in blocks:
            header_line = f"{'Variable':<{label_width}}" + "".join(
                f"{'Period ' + str(self.period[i]):>{col_width}}" for i in block)
            total_width = len(header_line)

            block_title = title
            if len(blocks) > 1:
                block_title += (f"  (periods {self.period[block[0]]}"
                                 f"-{self.period[block[-1]]})")

            print("=" * total_width)
            print(block_title)
            print("=" * total_width)
            print(header_line)
            print("-" * total_width)

            for name, values in variables:
                row_line = f"{name:<{label_width}}" + "".join(
                    f"{values[i]:>{col_width}.2f}" for i in block)
                print(row_line)

            print("=" * total_width)
            print()

    def printCurrentState(self):
        self._print_table([len(self.period) - 1], "CURRENT STATE \u2014 Model BMW (Chapter 7)")

    def printHistory(self):
        self._print_table(range(len(self.period)), "SIMULATION RESULTS \u2014 Model BMW (Chapter 7)")

    def checkConsistency(self, tol=1e-6):
        #Banks hold no net worth, so M_h must equal L every period.
        title = "STOCK-FLOW CONSISTENCY CHECK (M_h should equal L)"
        rows = []
        all_ok = True
        for i in range(1, len(self.period)):
            gap = self.Mh[i] - self.L[i]
            ok = abs(gap) < tol
            all_ok = all_ok and ok
            symbol = "\u2713" if ok else "\u2717"
            rows.append(f"Period {self.period[i]:<3} "
                        f"M_h = {self.Mh[i]:>10.2f}   L = {self.L[i]:>10.2f}   "
                        f"[{symbol}]")
        summary = ("All periods consistent." if all_ok
                   else "Inconsistency detected -- check the equations.")

        inner = max([len(title), len(summary)] + [len(r) for r in rows]) + 2
        top = "\u256d" + "\u2500" * inner + "\u256e"
        mid = "\u251c" + "\u2500" * inner + "\u2524"
        bottom = "\u2570" + "\u2500" * inner + "\u256f"

        print()
        print(top)
        print("\u2502" + title.center(inner) + "\u2502")
        print(mid)
        for row in rows:
            print("\u2502 " + row.ljust(inner - 1) + "\u2502")
        print(mid)
        print("\u2502" + summary.center(inner) + "\u2502")
        print(bottom)


def runEconomy(alpha0, alpha1, alpha2, delta, gamma, kappa, pr, r_bar,
               K_1, L_1, Mh_1, Y_1, r_1, n):
    econ = EconomyBMW(K_1, L_1, Mh_1, Y_1, r_1)
    for i in range(n):
        econ.runPeriod(alpha0[i], alpha1[i], alpha2[i], delta[i],
                        gamma[i], kappa[i], pr[i], r_bar[i])
    econ.printHistory()
    econ.checkConsistency()
    return econ


econ = runEconomy(
    alpha0=[25.0] * 2,
    alpha1=[0.75] * 2,
    alpha2=[0.05] * 2,
    delta=[0.10] * 2,
    gamma=[0.10] * 2,
    kappa=[1.0] * 2,
    pr=[1.0] * 2,
    r_bar=[0.04] * 2,
    K_1=0.0,
    L_1=0.0,
    Mh_1=0.0,
    Y_1=0.0,
    r_1=0.04,
    n=2,
)